Relational Database Auto-Ingestion & Upsert Engine
Goal: Ingest transformed data into a relational database with idempotency.
Build a database loader module that connects to SQLite or PostgreSQL, creates schemas, and performs upserts (insert or update on conflict) with transaction safety.

SQLAlchemy
SQLite
Python 3
Suggested Approach: Use sqlite3 or SQLAlchemy engine, execute batch inserts, and write query verification scripts.
Deliverable: Populated SQL database with structured tables, primary keys, and query verification scripts.

In [1]:
import sqlite3

# 1. Connect to SQLite database (creates 'quotes_database.db' file automatically)
conn = sqlite3.connect("quotes_database.db")
cursor = conn.cursor()

# 2. Create target schema with Primary Key
# Using 'quote_hash' or auto-increment 'id' as primary key
create_table_query = """
CREATE TABLE IF NOT EXISTS quotes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    quote TEXT NOT NULL UNIQUE,
    author TEXT NOT NULL,
    tags TEXT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

cursor.execute(create_table_query)
conn.commit()

print(" Database connected & schema created successfully!")

# Verify table creation
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables in database:", cursor.fetchall())

 Database connected & schema created successfully!
Tables in database: [('quotes',), ('sqlite_sequence',)]


In [3]:
import sqlite3
import json

# 1. Load the JSON data we scraped earlier (or create mock data if running standalone)
try:
    with open("quotes_100_records.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
except FileNotFoundError:
    # Fallback sample data if previous JSON isn't present
    raw_data = [
        {"quote": "The only limit to our realization of tomorrow is our doubts of today.", "author": "Franklin D. Roosevelt", "tags": "inspirational, hope"},
        {"quote": "Do what you can, with what you have, where you are.", "author": "Theodore Roosevelt", "tags": "motivational"}
    ]

# 2. Re-connect to database
conn = sqlite3.connect("quotes_database.db")
cursor = conn.cursor()

# 3. Format dataset into a list of tuples for SQL batch insertion
records_to_upsert = [
    (item["quote"], item["author"], item["tags"]) for item in raw_data
]

# 4. Write the Upsert SQL Query
upsert_query = """
INSERT INTO quotes (quote, author, tags)
VALUES (?, ?, ?)
ON CONFLICT(quote) DO UPDATE SET
    author = excluded.author,
    tags = excluded.tags,
    ingested_at = CURRENT_TIMESTAMP;
"""

# 5. Execute Batch Transaction with Safety
try:
    cursor.executemany(upsert_query, records_to_upsert)
    conn.commit()  # Save changes to disk
    print(f" Successfully upserted {len(records_to_upsert)} records into SQLite!")
except Exception as e:
    conn.rollback()  # Roll back transaction if an error occurs
    print(f" Transaction failed, changes rolled back. Error: {e}")
finally:
    conn.close()

 Successfully upserted 2 records into SQLite!


In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("quotes_database.db")

print("--- 1. VERIFICATION QUERIES ---")

# Query A: Check Total Row Count
total_rows = conn.execute("SELECT COUNT(*) FROM quotes").fetchone()[0]
print(f"Initial Total Records in Database: {total_rows}")

# Query B: Run an Aggregate Query (Top Authors by Quote Count)
top_authors_df = pd.read_sql_query("""
    SELECT author, COUNT(*) as quote_count
    FROM quotes
    GROUP BY author
    ORDER BY quote_count DESC
    LIMIT 5;
""", conn)

print("\nTop 5 Most Frequent Authors:")
print(top_authors_df)

# Query C: Inspect Recent Records with Auto-Generated Timestamps
sample_df = pd.read_sql_query("""
    SELECT id, quote, author, ingested_at
    FROM quotes
    LIMIT 3;
""", conn)

print("\nSample Ingested Rows (with Primary Key & Timestamp):")
print(sample_df)

print("\n--- 2. TESTING IDEMPOTENCY ---")

# Simulate re-running the exact same batch insertion from Step 2
duplicate_sample = [
    (sample_df.iloc[0]["quote"], "UPDATED AUTHOR TEST", "test_tag")
]

upsert_query = """
INSERT INTO quotes (quote, author, tags)
VALUES (?, ?, ?)
ON CONFLICT(quote) DO UPDATE SET
    author = excluded.author,
    tags = excluded.tags,
    ingested_at = CURRENT_TIMESTAMP;
"""

# Execute the upsert again with overlapping quote text
cursor = conn.cursor()
cursor.executemany(upsert_query, duplicate_sample)
conn.commit()

# Check row count after running duplicate ingestion
new_total_rows = conn.execute("SELECT COUNT(*) FROM quotes").fetchone()[0]
print(f"Total Records After Re-Ingestion: {new_total_rows}")

if total_rows == new_total_rows:
    print(" SUCCESS: Engine is Idempotent! Zero duplicate rows created.")
else:
    print(" FAILURE: Duplicates detected!")

conn.close()

--- 1. VERIFICATION QUERIES ---
Initial Total Records in Database: 2

Top 5 Most Frequent Authors:
                author  quote_count
0  UPDATED AUTHOR TEST            1
1   Theodore Roosevelt            1

Sample Ingested Rows (with Primary Key & Timestamp):
   id                                              quote               author  \
0   1  The only limit to our realization of tomorrow ...  UPDATED AUTHOR TEST   
1   2  Do what you can, with what you have, where you...   Theodore Roosevelt   

           ingested_at  
0  2026-10-26 09:37:20  
1  2026-10-26 09:36:25  

--- 2. TESTING IDEMPOTENCY ---
Total Records After Re-Ingestion: 2
 SUCCESS: Engine is Idempotent! Zero duplicate rows created.


In [ ]:
import json
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import create_engine, Column, Integer, Text, DateTime, MetaData, Table
from sqlalchemy.dialects.sqlite import insert as sqlite_upsert

# 1. Setup SQLAlchemy Database Engine
# (To switch to PostgreSQL, change URL to: "******localhost:5432/dbname")
DATABASE_URL = "sqlite:///sqlalchemy_quotes.db"
engine = create_engine(DATABASE_URL, echo=False)
metadata = MetaData()

# 2. Define Schema with Constraints (Using dynamic, timezone-aware UTC datetimes)
quotes_table = Table(
    "quotes",
    metadata,
    Column("id", Integer, primary_key=True, autoincrement=True),
    Column("quote", Text, nullable=False, unique=True),
    Column("author", Text, nullable=False),
    Column("tags", Text),
    Column(
        "ingested_at",
        DateTime,
        default=lambda: datetime.now(timezone.utc),
        onupdate=lambda: datetime.now(timezone.utc)
    )
)

# Create table in the database
metadata.create_all(engine)
print("SQLAlchemy Engine initialized and schema created.")

# 3. Load Scraped JSON Records
try:
    with open("quotes_100_records.json", "r", encoding="utf-8") as f:
        records = json.load(f)
except FileNotFoundError:
    records = [
        {"quote": "Integrity is doing the right thing, even when no one is watching.", "author": "C.S. Lewis", "tags": "ethics, integrity"}
    ]

# Format records into clean list of dicts
formatted_records = [
    {
        "quote": item["quote"],
        "author": item["author"],
        "tags": item["tags"] if isinstance(item["tags"], str) else ", ".join(item["tags"])
    }
    for item in records
]

# 4. Perform Idempotent Batch Upsert
def upsert_records(records_list):
    with engine.begin() as connection:  # Automatically opens transaction & commits on exit
        for record in records_list:
            current_time = datetime.now(timezone.utc)

            # Build SQLite ON CONFLICT DO UPDATE query
            stmt = sqlite_upsert(quotes_table).values(
                quote=record["quote"],
                author=record["author"],
                tags=record["tags"],
                ingested_at=current_time
            )

            upsert_stmt = stmt.on_conflict_do_update(
                index_elements=["quote"],  # Conflict target (UNIQUE constraint)
                set_={
                    "author": stmt.excluded.author,
                    "tags": stmt.excluded.tags,
                    "ingested_at": current_time
                }
            )

            connection.execute(upsert_stmt)

# Run Ingestion
upsert_records(formatted_records)
print(f"Ingested {len(formatted_records)} records using SQLAlchemy transaction manager.")

# 5. Run Verification Query with Pandas
df_verify = pd.read_sql("SELECT count(*) as total_rows FROM quotes", con=engine)
print(f"Verified Rows in Database: {df_verify.iloc[0]['total_rows']}")
